# Short Selling Screen

Standalone short-selling screen for the S&P 500, S&P 400, and S&P 600 universe.

This notebook is intentionally separate from `Equities.ipynb` because it performs large price-history and Yahoo metadata queries.


In [ ]:
# Block 2: Load libraries and initialize data clients

import logging
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False

import os
import sys
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

from dotenv import load_dotenv
from IPython.display import display
import numpy as np
import pandas as pd
import plotly.graph_objects as go

cwd = os.getcwd()
project_path = cwd
while not os.path.exists(os.path.join(project_path, "pyproject.toml")):
    parent = os.path.dirname(project_path)
    if parent == project_path:
        raise FileNotFoundError("Could not locate the project root from the current working directory.")
    project_path = parent
if project_path not in sys.path:
    sys.path.append(project_path)
load_dotenv(os.path.join(project_path, ".env"))

from Quantapp.data import GICSDataClient, MarketDataClient
from Quantapp.data import yf as qa_yf

warnings.filterwarnings("ignore")

md = MarketDataClient()
gics_data = GICSDataClient(save_path=project_path)


In [ ]:
# Block 3: Build lightweight all-company metadata table

all_companies = gics_data.retrieve_companies()

index_by_capitalization = {
    'Large Cap': 'S&P 500',
    'Mid Cap': 'S&P 400',
    'Small Cap': 'S&P 600',
}

local_market_cap_cache_path = os.path.join(project_path, 'company_data', 'fmp_historical_market_caps')

def read_last_nonempty_line(path, chunk_size=4096):
    with open(path, 'rb') as file:
        file.seek(0, os.SEEK_END)
        position = file.tell()
        buffer = b''

        while position > 0:
            read_size = min(chunk_size, position)
            position -= read_size
            file.seek(position)
            buffer = file.read(read_size) + buffer
            lines = [line.strip() for line in buffer.splitlines() if line.strip()]
            if len(lines) > 1 or position == 0:
                return lines[-1].decode('utf-8')

    return ''

def load_latest_cached_market_caps(symbols, cache_path=local_market_cap_cache_path):
    latest_market_caps = {}
    if not os.path.exists(cache_path):
        return pd.Series(pd.NA, index=pd.Index(symbols, name='Symbol'), name='Market Cap')

    for symbol in symbols:
        symbol = str(symbol)
        market_cap_file = os.path.join(cache_path, f'{symbol}.csv')
        if not os.path.exists(market_cap_file):
            latest_market_caps[symbol] = pd.NA
            continue

        try:
            latest_line = read_last_nonempty_line(market_cap_file)
            latest_market_caps[symbol] = pd.to_numeric(latest_line.split(',')[-1], errors='coerce')
        except Exception:
            latest_market_caps[symbol] = pd.NA

    return pd.Series(latest_market_caps, name='Market Cap')

all_company_metadata_table = all_companies.copy()

if 'Market Cap' in all_company_metadata_table.columns:
    all_company_metadata_table['Market Cap'] = pd.to_numeric(all_company_metadata_table['Market Cap'], errors='coerce')
else:
    cached_market_caps = load_latest_cached_market_caps(all_company_metadata_table['Symbol'].astype(str).tolist())
    all_company_metadata_table['Market Cap'] = pd.to_numeric(
        all_company_metadata_table['Symbol'].map(cached_market_caps),
        errors='coerce',
    )

all_company_metadata_table['Index'] = all_company_metadata_table['Capitalization'].map(index_by_capitalization)
all_company_metadata_table = all_company_metadata_table.set_index('Symbol').loc[
    :,
    ['Index', 'Capitalization', 'Market Cap', 'Sector', 'Industry Group', 'Industry', 'Sub-Industry'],
]


In [ ]:
# Block 4: Build short-selling return-risk table

if 'all_company_metadata_table' not in globals():
    raise NameError('Run Block 5 first to build all_company_metadata_table.')

return_scan_period = '3y'
return_scan_interval = '1d'
return_scan_batch_size = 150
return_scan_retry_count = 3
return_scan_retry_pause_seconds = 5
return_windows = {
    '1D': 1,
    '1W': 5,
    '1M': 21,
}
largest_up_move_columns = {
    '1D': 'Largest 1D Up Move 3Y (%)',
    '1W': 'Largest 1W Up Move 3Y (%)',
    '1M': 'Largest 1M Up Move 3Y (%)',
}
latest_return_columns = {
    '1D': "Today's Return (%)",
    '1W': 'Past Week Return (%)',
    '1M': 'Past Month Return (%)',
}
todays_return_zscore_column = "Today's Return Z-Score"
price_columns = {
    'highest': 'Highest Price 3Y',
    'lowest': 'Lowest Price 3Y',
    'average': 'Average Price 3Y',
    'current': 'Current Price',
}
current_vs_average_price_column = 'Current vs Avg Price 3Y (%)'
import time

short_interest_max_workers = 2
short_interest_retry_count = 3
short_interest_retry_pause_seconds = 5
short_interest_request_pause_seconds = 0.25
short_interest_cache_ttl_hours = 168
short_interest_cache_path = os.path.join(project_path, 'company_data', 'yfinance_short_interest_cache.csv')
short_interest_fields = {
    'sharesShort': 'Shares Short',
    'sharesShortPriorMonth': 'Shares Short Prior Month',
    'sharesShortPreviousMonthDate': 'Shares Short Prior Month Date',
    'dateShortInterest': 'Short Interest Date',
    'shortPercentOfFloat': 'Short % Float',
    'sharesPercentSharesOut': 'Short % Shares Outstanding',
    'shortRatio': 'Short Ratio',
}
short_interest_extra_fields = {
    'floatShares': 'Float Shares',
    'sharesOutstanding': 'Shares Outstanding',
    'averageVolume': 'Average Volume',
}
short_interest_fetched_at_column = 'Short Interest Fetched At'
short_interest_status_column = 'Short Interest Data Status'
short_interest_missing_fields_column = 'Short Interest Missing Fields'
short_interest_date_columns = ['Shares Short Prior Month Date', 'Short Interest Date']
short_interest_percent_columns = ['Short % Float', 'Short % Shares Outstanding']
short_interest_output_columns = list(short_interest_fields.values()) + [
    short_interest_status_column,
    short_interest_missing_fields_column,
]

def load_close_prices_in_batches(symbols, batch_size=return_scan_batch_size):
    price_frames = []
    symbols = [str(symbol) for symbol in symbols]

    for start in range(0, len(symbols), batch_size):
        batch_symbols = symbols[start:start + batch_size]
        batch_prices = pd.DataFrame()

        for attempt in range(return_scan_retry_count):
            try:
                batch_prices = md.generate_series(
                    batch_symbols,
                    columns=['Close'],
                    period=return_scan_period,
                    interval=return_scan_interval,
                )
                break
            except Exception as exc:
                if attempt >= return_scan_retry_count - 1:
                    print(f'Price fetch failed for batch starting at {start}: {exc}')
                else:
                    time.sleep(return_scan_retry_pause_seconds * (attempt + 1))

        if isinstance(batch_prices, pd.Series):
            batch_prices = batch_prices.to_frame(name=batch_prices.name or batch_symbols[0])

        if isinstance(batch_prices, pd.DataFrame) and not batch_prices.empty:
            price_frames.append(batch_prices)

    if not price_frames:
        return pd.DataFrame(index=pd.DatetimeIndex([]), columns=symbols, dtype=float)

    close_prices = pd.concat(price_frames, axis=1)
    close_prices = close_prices.loc[:, ~close_prices.columns.duplicated()]
    close_prices = close_prices.reindex(columns=symbols)
    return close_prices.apply(pd.to_numeric, errors='coerce').sort_index()

def empty_return_series(symbols):
    return pd.Series(np.nan, index=pd.Index(symbols, name='Symbol'), dtype=float)

def largest_positive_return_percent(return_frame, symbols):
    if return_frame.empty:
        return empty_return_series(symbols)

    return (
        return_frame.where(return_frame > 0)
        .max(axis=0)
        .mul(100)
        .reindex(symbols)
    )

def latest_return_percent(return_frame, symbols):
    if return_frame.empty:
        return empty_return_series(symbols)

    latest_row = return_frame.dropna(how='all').tail(1)
    if latest_row.empty:
        return empty_return_series(symbols)

    return latest_row.iloc[-1].mul(100).reindex(symbols)

def normalize_yahoo_symbol(symbol):
    return str(symbol).replace('.', '-')

def is_missing_value(value):
    return value is None or pd.isna(value)

def coerce_snapshot_number(snapshot, column):
    return pd.to_numeric(snapshot.get(column), errors='coerce')

def complete_short_interest_snapshot(snapshot):
    snapshot = dict(snapshot or {})
    shares_short = coerce_snapshot_number(snapshot, 'Shares Short')
    float_shares = coerce_snapshot_number(snapshot, 'Float Shares')
    shares_outstanding = coerce_snapshot_number(snapshot, 'Shares Outstanding')
    average_volume = coerce_snapshot_number(snapshot, 'Average Volume')

    if is_missing_value(snapshot.get('Short % Float')) and pd.notna(shares_short) and pd.notna(float_shares) and float_shares != 0:
        snapshot['Short % Float'] = shares_short / float_shares

    if is_missing_value(snapshot.get('Short % Shares Outstanding')) and pd.notna(shares_short) and pd.notna(shares_outstanding) and shares_outstanding != 0:
        snapshot['Short % Shares Outstanding'] = shares_short / shares_outstanding

    if is_missing_value(snapshot.get('Short Ratio')) and pd.notna(shares_short) and pd.notna(average_volume) and average_volume != 0:
        snapshot['Short Ratio'] = shares_short / average_volume

    missing_fields = [
        column for column in short_interest_fields.values()
        if column not in short_interest_date_columns and is_missing_value(snapshot.get(column))
    ]
    missing_fields.extend(
        column for column in short_interest_date_columns
        if pd.isna(pd.to_datetime(snapshot.get(column), errors='coerce'))
    )

    existing_status = snapshot.get(short_interest_status_column)
    if existing_status not in ['Rate Limited', 'Fetch Error']:
        snapshot[short_interest_status_column] = 'Complete' if not missing_fields else 'Partial'
    snapshot[short_interest_missing_fields_column] = ', '.join(dict.fromkeys(missing_fields))
    return snapshot

def load_persistent_short_interest_cache():
    if not os.path.exists(short_interest_cache_path):
        return {}

    try:
        cache_frame = pd.read_csv(short_interest_cache_path)
    except Exception:
        return {}

    if 'Symbol' not in cache_frame.columns:
        return {}

    cache = {}
    for _, row in cache_frame.iterrows():
        symbol = str(row['Symbol'])
        snapshot = row.drop(labels=['Symbol']).to_dict()
        cache[symbol] = complete_short_interest_snapshot(snapshot)
    return cache

def save_persistent_short_interest_cache(cache):
    if not cache:
        return

    os.makedirs(os.path.dirname(short_interest_cache_path), exist_ok=True)
    cache_frame = pd.DataFrame.from_dict(cache, orient='index')
    cache_frame.index.name = 'Symbol'
    cache_frame.reset_index().to_csv(short_interest_cache_path, index=False)

def short_interest_cache_is_fresh(snapshot):
    fetched_at = pd.to_datetime(snapshot.get(short_interest_fetched_at_column), errors='coerce', utc=True)
    if pd.isna(fetched_at):
        return False

    cache_age_hours = (pd.Timestamp.utcnow() - fetched_at).total_seconds() / 3600
    return cache_age_hours <= short_interest_cache_ttl_hours

def fetch_short_interest_snapshot(symbol):
    snapshot = {
        column_name: np.nan
        for column_name in list(short_interest_fields.values()) + list(short_interest_extra_fields.values())
    }
    last_error = ''

    for attempt in range(short_interest_retry_count):
        if short_interest_request_pause_seconds:
            time.sleep(short_interest_request_pause_seconds)

        try:
            info = qa_yf.Ticker(normalize_yahoo_symbol(symbol)).info or {}
            for field_name, column_name in short_interest_fields.items():
                snapshot[column_name] = info.get(field_name)
            for field_name, column_name in short_interest_extra_fields.items():
                snapshot[column_name] = info.get(field_name)
            snapshot[short_interest_fetched_at_column] = pd.Timestamp.utcnow().isoformat()
            snapshot[short_interest_status_column] = 'Fetched'
            break
        except Exception as exc:
            last_error = str(exc)
            error_label = type(exc).__name__
            rate_limited = 'RateLimit' in error_label or 'Too Many Requests' in last_error or 'rate limit' in last_error.lower()
            if attempt < short_interest_retry_count - 1:
                pause_seconds = short_interest_retry_pause_seconds * (attempt + 1 if rate_limited else 0.5)
                time.sleep(pause_seconds)
                continue

            snapshot[short_interest_fetched_at_column] = pd.Timestamp.utcnow().isoformat()
            snapshot[short_interest_status_column] = 'Rate Limited' if rate_limited else 'Fetch Error'
            snapshot[short_interest_missing_fields_column] = last_error[:200]

    return str(symbol), complete_short_interest_snapshot(snapshot)

def load_short_interest_snapshots(symbols):
    symbols = [str(symbol) for symbol in symbols]
    if 'short_interest_snapshot_cache' not in globals():
        globals()['short_interest_snapshot_cache'] = load_persistent_short_interest_cache()

    cache = globals()['short_interest_snapshot_cache']
    missing_symbols = [
        symbol for symbol in symbols
        if symbol not in cache or not short_interest_cache_is_fresh(cache[symbol])
    ]

    if missing_symbols:
        max_workers = min(short_interest_max_workers, max(1, len(missing_symbols)))
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            for symbol, snapshot in executor.map(fetch_short_interest_snapshot, missing_symbols):
                cache[symbol] = snapshot
        save_persistent_short_interest_cache(cache)

    short_interest_table = pd.DataFrame.from_dict(
        {symbol: cache.get(symbol, {}) for symbol in symbols},
        orient='index',
    ).reindex(symbols)

    for column in list(short_interest_fields.values()) + list(short_interest_extra_fields.values()) + [short_interest_fetched_at_column, short_interest_status_column, short_interest_missing_fields_column]:
        if column not in short_interest_table.columns:
            short_interest_table[column] = np.nan

    numeric_columns = [
        column for column in list(short_interest_fields.values()) + list(short_interest_extra_fields.values())
        if column not in short_interest_date_columns
    ]
    short_interest_table[numeric_columns] = short_interest_table[numeric_columns].apply(
        pd.to_numeric,
        errors='coerce',
    )

    for column in short_interest_date_columns:
        short_interest_table[column] = pd.to_datetime(
            short_interest_table[column],
            unit='s',
            errors='coerce',
        )

    return short_interest_table.loc[:, short_interest_output_columns]

short_selling_symbols = all_company_metadata_table.index.astype(str).tolist()
scan_close_prices = load_close_prices_in_batches(short_selling_symbols)
short_interest_table = load_short_interest_snapshots(short_selling_symbols)

for column in short_interest_table.columns:
    all_company_metadata_table[column] = short_interest_table[column]

scan_price_frame = scan_close_prices

return_frames = {
    label: scan_close_prices.pct_change(periods=trading_days, fill_method=None)
    for label, trading_days in return_windows.items()
}

for label, column_name in largest_up_move_columns.items():
    all_company_metadata_table[column_name] = largest_positive_return_percent(
        return_frames[label],
        short_selling_symbols,
    )

for label, column_name in latest_return_columns.items():
    all_company_metadata_table[column_name] = latest_return_percent(
        return_frames[label],
        short_selling_symbols,
    )

all_company_metadata_table[price_columns['highest']] = scan_price_frame.max(axis=0).reindex(
    short_selling_symbols
)
all_company_metadata_table[price_columns['lowest']] = scan_price_frame.min(axis=0).reindex(
    short_selling_symbols
)
all_company_metadata_table[price_columns['average']] = scan_price_frame.mean(axis=0).reindex(
    short_selling_symbols
)

latest_close_row = scan_close_prices.dropna(how='all').tail(1)
if latest_close_row.empty:
    all_company_metadata_table[price_columns['current']] = empty_return_series(short_selling_symbols)
else:
    all_company_metadata_table[price_columns['current']] = latest_close_row.iloc[-1].reindex(
        short_selling_symbols
    )

average_price_series = all_company_metadata_table[price_columns['average']]
current_price_series = all_company_metadata_table[price_columns['current']]
all_company_metadata_table[current_vs_average_price_column] = (
    current_price_series.sub(average_price_series)
    .div(average_price_series.replace(0, np.nan))
    .mul(100)
)

todays_return_series = all_company_metadata_table[latest_return_columns['1D']]
todays_return_std = todays_return_series.std(skipna=True, ddof=0)

if pd.notna(todays_return_std) and todays_return_std != 0:
    all_company_metadata_table[todays_return_zscore_column] = (
        todays_return_series - todays_return_series.mean(skipna=True)
    ) / todays_return_std
else:
    all_company_metadata_table[todays_return_zscore_column] = np.nan

short_selling_return_columns = list(largest_up_move_columns.values()) + list(latest_return_columns.values())
all_company_metadata_table[short_selling_return_columns] = all_company_metadata_table[
    short_selling_return_columns
].round(2)
all_company_metadata_table[current_vs_average_price_column] = all_company_metadata_table[
    current_vs_average_price_column
].round(2)
all_company_metadata_table[short_interest_percent_columns] = all_company_metadata_table[
    short_interest_percent_columns
].mul(100).round(2)
all_company_metadata_table['Short Ratio'] = all_company_metadata_table['Short Ratio'].round(2)
all_company_metadata_table[todays_return_zscore_column] = all_company_metadata_table[
    todays_return_zscore_column
].round(2)
short_selling_price_columns = list(price_columns.values())
all_company_metadata_table[short_selling_price_columns] = all_company_metadata_table[
    short_selling_price_columns
].round(2)

if 'Market Cap' in all_company_metadata_table.columns:
    all_company_metadata_table['Market Cap'] = all_company_metadata_table['Market Cap'].round(0)

short_selling_default_sort_column = latest_return_columns['1D']
short_selling_company_table = all_company_metadata_table.sort_values(
    short_selling_default_sort_column,
    ascending=False,
    na_position='last',
)

plotly_percent_columns = short_selling_return_columns + [current_vs_average_price_column] + short_interest_percent_columns
plotly_zscore_columns = [todays_return_zscore_column]
plotly_price_columns = short_selling_price_columns
plotly_integer_columns = ['Shares Short', 'Shares Short Prior Month']
plotly_date_columns = short_interest_date_columns
plotly_numeric_sort_columns = set(
    plotly_percent_columns
    + plotly_zscore_columns
    + plotly_price_columns
    + plotly_integer_columns
    + ['Market Cap', 'Short Ratio']
)
plotly_sort_source_table = all_company_metadata_table.reset_index().copy()

def format_short_selling_plotly_table(table):
    formatted_table = table.copy()

    if 'Market Cap' in formatted_table.columns:
        formatted_table['Market Cap'] = formatted_table['Market Cap'].map(
            lambda value: f'${value:,.0f}' if pd.notna(value) else ''
        )

    for column in plotly_percent_columns:
        formatted_table[column] = formatted_table[column].map(
            lambda value: f'{value:.2f}%' if pd.notna(value) else ''
        )

    for column in plotly_zscore_columns:
        formatted_table[column] = formatted_table[column].map(
            lambda value: f'{value:.2f}' if pd.notna(value) else ''
        )

    for column in plotly_price_columns:
        formatted_table[column] = formatted_table[column].map(
            lambda value: f'${value:,.2f}' if pd.notna(value) else ''
        )

    for column in plotly_integer_columns:
        formatted_table[column] = formatted_table[column].map(
            lambda value: f'{value:,.0f}' if pd.notna(value) else ''
        )

    if 'Short Ratio' in formatted_table.columns:
        formatted_table['Short Ratio'] = formatted_table['Short Ratio'].map(
            lambda value: f'{value:.2f}' if pd.notna(value) else ''
        )

    for column in plotly_date_columns:
        formatted_table[column] = formatted_table[column].map(
            lambda value: value.strftime('%Y-%m-%d') if pd.notna(value) else ''
        )

    return formatted_table

def plotly_table_values(table):
    return [table[column].tolist() for column in table.columns]

def sort_for_plotly_table(column):
    ascending = column not in plotly_numeric_sort_columns
    sorted_table = plotly_sort_source_table.sort_values(
        column,
        ascending=ascending,
        na_position='last',
        kind='mergesort',
    )
    return sorted_table, ascending

plotly_short_selling_table = format_short_selling_plotly_table(
    short_selling_company_table.reset_index().copy()
)

plotly_sort_columns = plotly_short_selling_table.columns.tolist()
plotly_sort_columns = [
    short_selling_default_sort_column,
    *[column for column in plotly_sort_columns if column != short_selling_default_sort_column],
]
plotly_sort_buttons = []

for column in plotly_sort_columns:
    sorted_table, ascending = sort_for_plotly_table(column)
    formatted_table = format_short_selling_plotly_table(sorted_table)
    direction_label = 'Ascending' if ascending else 'Descending'
    plotly_sort_buttons.append(
        dict(
            label=f'{column} ({direction_label})',
            method='update',
            args=[
                {'cells.values': [plotly_table_values(formatted_table)]},
                {'title.text': f'Short-Selling Screen Sorted by {column} ({direction_label})'},
            ],
        )
    )

short_selling_plotly_table = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=plotly_short_selling_table.columns.tolist(),
                fill_color='#0f172a',
                font=dict(color='white', size=12),
                align='left',
                height=28,
            ),
            cells=dict(
                values=[plotly_short_selling_table[column] for column in plotly_short_selling_table.columns],
                fill_color='#ffffff',
                font=dict(color='#0f172a', size=11),
                align='left',
                height=24,
            ),
        )
    ]
)
short_selling_plotly_table.update_layout(
    title=f'Short-Selling Screen Sorted by {short_selling_default_sort_column}',
    height=900,
    margin=dict(l=12, r=12, t=92, b=12),
    updatemenus=[
        dict(
            buttons=plotly_sort_buttons,
            direction='down',
            showactive=True,
            active=0,
            x=0,
            xanchor='left',
            y=1.08,
            yanchor='top',
        )
    ],
)

short_selling_plotly_table.show()
